In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
df = pd.read_csv('/content/drive/Shareddrives/230 Data Viz Project/fire_nrt_J1V-C2_725729.csv')
df.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-18.99333,-174.76727,338.14,0.61,0.71,2026-01-01,26,N20,VIIRS,n,2.0NRT,284.34,12.02,D
1,-18.99204,-174.76611,342.67,0.61,0.71,2026-01-01,26,N20,VIIRS,n,2.0NRT,286.74,9.88,D
2,67.55039,83.20029,348.26,0.73,0.76,2026-01-01,58,N20,VIIRS,n,2.0NRT,249.40,8.95,N
3,71.19350,67.09715,321.12,0.61,0.53,2026-01-01,58,N20,VIIRS,n,2.0NRT,255.69,3.41,N
4,68.52409,79.94820,320.47,0.59,0.70,2026-01-01,58,N20,VIIRS,n,2.0NRT,248.18,4.18,N


In [26]:
df.isnull().sum()

,0
latitude,0
longitude,0
brightness,0
scan,0
track,0
acq_date,0
acq_time,0
satellite,0
instrument,0
confidence,0


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1013206 entries, 0 to 1013205
Data columns (total 14 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   latitude    1013206 non-null  float64
 1   longitude   1013206 non-null  float64
 2   brightness  1013206 non-null  float64
 3   scan        1013206 non-null  float64
 4   track       1013206 non-null  float64
 5   acq_date    1013206 non-null  object 
 6   acq_time    1013206 non-null  int64  
 7   satellite   1013206 non-null  object 
 8   instrument  1013206 non-null  object 
 9   confidence  1013206 non-null  object 
 10  version     1013206 non-null  object 
 11  bright_t31  1013206 non-null  float64
 12  frp         1013206 non-null  float64
 13  daynight    1013206 non-null  object 
dtypes: float64(7), int64(1), object(6)
memory usage: 108.2+ MB


## Feature Transformation based on our EDA

### Log transforming frp

In [28]:
df['frp'] = np.log1p(df['frp'])

### Date and time features

In [29]:
df['acq_date'] = pd.to_datetime(df['acq_date'])
df['month'] = df['acq_date'].dt.month

### Encoding categorical variables

In [30]:
df['confidence'] = df['confidence'].map({'l': 0, 'n': 1, 'h': 2})
df['is_day'] = (df['daynight'] == 'D').astype(int)

### Dropping redundant columns

In [31]:
df.drop(columns=['version', 'daynight', 'acq_date', 'satellite', 'instrument'], inplace=True)

In [32]:
df.head()

,latitude,longitude,brightness,scan,track,acq_time,confidence,bright_t31,frp,month,is_day
0,-18.99333,-174.76727,338.14,0.61,0.71,26,1,284.34,2.566487,1,1
1,-18.99204,-174.76611,342.67,0.61,0.71,26,1,286.74,2.386926,1,1
2,67.55039,83.20029,348.26,0.73,0.76,58,1,249.40,2.297573,1,0
3,71.19350,67.09715,321.12,0.61,0.53,58,1,255.69,1.483875,1,0
4,68.52409,79.94820,320.47,0.59,0.70,58,1,248.18,1.644805,1,0


## Splitting: 80-20 because Grid Search CV does validation splits

In [33]:
X = df.drop(columns=['frp'])
y = df['frp']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Training model - Linear Regression

In [34]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

### Test

In [35]:
y_test_pred = lr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)
print(f"Linear Regression - RMSE: {rmse:.4f}, R²: {r2:.4f}")

Linear Regression - RMSE: 0.5600, R²: 0.5216


### Grid Search CV does Cross Validation. So, we do not need a separate validation split. Combining train and val

#### Using ridge regularization for grid search

In [36]:
pipe = Pipeline([
    ('pca', PCA()),
    ('ridge', Ridge())
])

param_grid = {
    'pca__n_components': [2, 3, 5, 7, None],
    'ridge__alpha': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 25 candidates, totalling 75 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('pca', PCA()), ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'pca__n_components': [2, 3, 5, 7, None],
                         'ridge__alpha': [0.01, 0.1, 1, 10, 100]},
             scoring='r2', verbose=2)

In [37]:
print("Best alpha:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

Best alpha: {'pca__n_components': None, 'ridge__alpha': 0.1}
Best CV R²: 0.5193168757757226


### Eval on test

In [38]:
best_model = grid.best_estimator_

y_test_pred = best_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("Test Results")
print("RMSE:", test_rmse)
print("R²:", test_r2)

Test Results
RMSE: 0.5600345125487657
R²: 0.5215753170647696


## XGBoost

In [39]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,

)

xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=-1, num_parallel_tree=None, ...)

In [40]:
y_test_pred = xgb.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("XGBoost Test Results")
print("RMSE:", test_rmse)
print("R²:", test_r2)

XGBoost Test Results
RMSE: 0.4768076188116049
R²: 0.6532069644545102


### With grid search cv

In [42]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    'xgb__n_estimators': [50, 100],
    'xgb__max_depth': [4, 6],
    'xgb__learning_rate': [0.05, 0.1],
    'xgb__subsample': [0.8],
    'xgb__colsample_bytree': [0.8]
}

grid_xgb = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid_xgb.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('xgb',
                                        XGBRegressor(base_score=None,
                                                     booster=None,
                                                     callbacks=None,
                                                     colsample_bylevel=None,
                                                     colsample_bynode=None,
                                                     colsample_bytree=None,
                                                     device=None,
                                                     early_stopping_rounds=None,
                                                     enable_categorical=False,
                                                     eval_metric=None,
                                                     feature_types=None,
                                                     feature_weights=None,
                                                     gamma=None,
                                                     grow_policy=Non...
                                                     max_depth=None,
                                                     max_leaves=None,
                                                     min_child_weight=None,
                                                     missing=nan,
                                                     monotone_constraints=None,
                                                     multi_strategy=None,
                                                     n_estimators=None,
                                                     n_jobs=-1,
                                                     num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'xgb__colsample_bytree': [0.8],
                         'xgb__learning_rate': [0.05, 0.1],
                         'xgb__max_depth': [4, 6],
                         'xgb__n_estimators': [50, 100],
                         'xgb__subsample': [0.8]},
             scoring='r2', verbose=2)

In [43]:
print("Best Parameters:", grid_xgb.best_params_)
print("Best CV R²:", grid_xgb.best_score_)

Best Parameters: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 6, 'xgb__n_estimators': 100, 'xgb__subsample': 0.8}
Best CV R²: 0.6509464045054373


In [44]:
best_xgb = grid_xgb.best_estimator_

y_test_pred = best_xgb.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("XGBoost Test Results")
print("RMSE:", test_rmse)
print("R²:", test_r2)

XGBoost Test Results
RMSE: 0.4768076188116049
R²: 0.6532069644545102


### Gradient Boost

In [45]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train, y_train)

y_test_pred = gbr.predict(X_test)

In [46]:
y_test_pred = gbr.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("Test Results (Log Scale)")
print("RMSE:", test_rmse)
print("R²:", test_r2)

Test Results (Log Scale)
RMSE: 0.49236311146828743
R²: 0.6302101302382865


In [47]:
y_test_pred_real = np.expm1(y_test_pred)
y_test_real = np.expm1(y_test)

test_rmse_real = np.sqrt(mean_squared_error(y_test_real, y_test_pred_real))

print("\nTest Results (Original FRP Scale)")
print("RMSE:", test_rmse_real)


Test Results (Original FRP Scale)
RMSE: 13.533752407567416


In [48]:
import joblib

joblib.dump(best_xgb, 'gb_model.pkl')   # or rf, gbr, lr

['gb_model.pkl']